# 4. 工具的应用案例

## 4.1 案例1：使用args_schema

arg_schema给出明确的参数信息

模型初始化:

In [2]:
import asyncio
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# 从.env文件中加载环境变量
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model = ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

工具的定义与模拟调用：

In [7]:
from pydantic import BaseModel, Field
from langchain.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
from langchain.messages import HumanMessage

class WeatherSchema(BaseModel):
    city: str = Field(
        default= "北京",
        description="城市名称"
    )
    if_forecast: bool = Field(
        default= False,
        description="是否包含明日天气预报"
    )
    
@tool("get_weather_and_forecast", description="查询当日天气，可以包含明日天气预报", args_schema=WeatherSchema)
def get_weather(city: str, if_forecast: bool = False):
    res = f"{city} 今天天气不错"
    if if_forecast:
        res += "\n明天也不错"
    return res

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather_and_forecast',
        'description': '查询当日天气，可以包含明日天气预报',
        'parameters': {
            'properties': {
                'city': {'default': '北京', 'description': '城市名称', 'type': 'string'},
                'if_forecast': {'default': False, 'description': '是否包含明日天气预报', 'type': 'boolean'}
            },
            'type': 'object'
        }
    }
}

In [8]:
model_with_tools = model.bind_tools([get_weather])

messages = [HumanMessage("今天杭州天气如何？明天呢？")]
response = model_with_tools.invoke(messages)
messages.append(response)
tool_calls = response.tool_calls


for tool_call in tool_calls:
    if tool_call["name"] == "get_weather_and_forecast":
        tool_msg = get_weather.invoke(tool_call)
        messages.append(tool_msg)
final_response = model_with_tools.invoke(messages)
messages.append(final_response)
for msg in messages:
    msg.pretty_print()

================================ Human Message =================================

今天杭州天气如何？明天呢？
================================== Ai Message ==================================

[{'arguments': '{"city":"杭州","if_forecast":true}', 'call_id': 'call_fyDSY6hLbeUtbkKunUydVhh3', 'name': 'get_weather_and_forecast', 'type': 'function_call', 'id': 'fc_0798b9dcd9a412da016ab77d3393e887d29457ef3a47a5e07e', 'status': 'completed', 'internal_chat_message_metadata_passthrough': {'create_time': 1790410034.990655, 'turn_id': '01a0dcc1-07e1-776a-bb94-c0ab3dfd6ac1'}, 'metadata': {'turn_id': '01a0dcc1-07e1-776a-bb94-c0ab3dfd6ac1'}}]
Tool Calls:
  get_weather_and_forecast (call_fyDSY6hLbeUtbkKunUydVhh3)
 Call ID: call_fyDSY6hLbeUtbkKunUydVhh3
  Args:
    city: 杭州
    if_forecast: True
================================= Tool Message =================================
Name: get_weather_and_forecast

杭州 今天天气不错
明天也不错
================================== Ai Message ==================================

[{'type': 'text'

## 4.2 案例2：撰写docstring

可以在docstring中撰写参数的描述信息，此时参数默认值和类型都要通过函数签名传递。

模型初始化：

In [9]:
import asyncio
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# 从.env文件中加载环境变量
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model = ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

In [10]:
@tool("get_weather_and_forecast", parse_docstring=True)
def get_weather(city: str="北京", if_forecast: bool=False):
    """查询当日天气，可以包含明日天气预报
    
    Args:
        city: 城市名称
        if_forecast: 是否包含明日天气预报"""
    res = f"{city} 今天天气不错"
    if if_forecast:
        res += "\n明天要下雨"
    return res

model_with_tools = model.bind_tools([get_weather])

messages = [HumanMessage("今天杭州天气如何？明天呢？")]
response = model_with_tools.invoke(messages)
messages.append(response)
tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather_and_forecast":
        tool_msg = get_weather.invoke(tool_call)
        messages.append(tool_msg)
final_response = model_with_tools.invoke(messages)
messages.append(final_response)
for msg in messages:
    msg.pretty_print()


================================ Human Message =================================

今天杭州天气如何？明天呢？
================================== Ai Message ==================================

[{'arguments': '{"city":"杭州","if_forecast":true}', 'call_id': 'call_RvaeCJShXh9tNr2EASJzxGPc', 'name': 'get_weather_and_forecast', 'type': 'function_call', 'id': 'fc_0342ce60e1e700a5016ab77ed6aefc87d2bfa95a20f034a5e8', 'status': 'completed', 'internal_chat_message_metadata_passthrough': {'create_time': 1790410453.978766, 'turn_id': '01a0dcc7-6fc8-7cf4-a5b1-eb4ef7fe666f'}, 'metadata': {'turn_id': '01a0dcc7-6fc8-7cf4-a5b1-eb4ef7fe666f'}}]
Tool Calls:
  get_weather_and_forecast (call_RvaeCJShXh9tNr2EASJzxGPc)
 Call ID: call_RvaeCJShXh9tNr2EASJzxGPc
  Args:
    city: 杭州
    if_forecast: True
================================= Tool Message =================================
Name: get_weather_and_forecast

杭州 今天天气不错
明天要下雨
================================== Ai Message ==================================

[{'type': 'text'

## 4.3 多工具调用

大模型调用工具是单次推理，即每次运行调用一个工具，当调用多个工具时，需要用户自己管理多次调
用循环。

In [12]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool

# 1.定义工具
# 定义股票查询工具
@tool(parse_docstring=True)
def get_stock_price(company: str,timeframe: str="today") -> str:
    """查询指定公司的股票价格
    
    Args:
        company: 公司名称(如: 苹果公司，微软公司，谷歌公司)
        timeframe: 时间范围，默认为 today

    """
    # 模拟股票数据
    mock_data = {
        "苹果公司": {"today": 185.20, "week": 183.50, "month": 180.75},
        "微软公司": {"today": 415.86, "week": 412.30, "month": 405.42},
        "谷歌公司": {"today": 15.42, "week": 15.20, "month": 14.85}
    }

    if company not in mock_data:
        return f"公司 {company} 不存在"
    else:
        price = mock_data[company][timeframe]
        return f"{company} {timeframe} 价格为 {price}美元"


# 定义新闻搜索工具
@tool(parse_docstring=True)
def search_news(company: str) -> str:
    """搜索指定公司的财经新闻
    
    Args:
        company: 公司名称

    Returns:
        公司的财经新闻，每个新闻占一行

    """
    # 模拟新闻数据
    mock_news = {
        "苹果公司": [
            "苹果发布新款iPhone，股价上涨3%",
            "苹果与欧盟达成反垄断和解协议",
            "苹果将在印度扩大生产规模"
        ],
        "微软公司": [
            "微软Azure云业务季度增长超预期",
            "微软完成对Nuance的收购",
            "微软推出新一代AI助手Copilot"
        ],
        "谷歌公司": [
            "谷歌发布新AI模型，性能提升20%",
            "谷歌与OpenAI合作，开发新的AI助手",
            "谷歌在欧洲展开AI研究项目"
        ]
    }

    news_list = mock_news.get(company, [f"未找到{company}的相关新闻"])
    return "\n".join(news_list)
        
rprint(convert_to_openai_tool(search_news))


{
    'type': 'function',
    'function': {
        'name': 'search_news',
        'description': '搜索指定公司的财经新闻',
        'parameters': {
            'properties': {'company': {'description': '公司名称', 'type': 'string'}},
            'required': ['company'],
            'type': 'object'
        }
    }
}

In [16]:
# 工具调用

tools = [search_news, get_stock_price]
model_with_tools = model.bind_tools(tools)

messages_list = []
human_message = HumanMessage(content="苹果公司今天的股价是多少？最近有什么新闻？")
messages_list.append(human_message)

# 3. 工具调用
while True:
    response = model_with_tools.invoke(messages_list)
    messages_list.append(response)
    if not response.tool_calls:
        print("没有工具调用，直接返回答案")
        break

    # 如果有工具调用，执行工具调用
    for tool_call in response.tool_calls:
        tool_name = tool_call["name"]
        #tool_args = tool_call["arguments"]
        if tool_name == "search_news":
            news_result = search_news.invoke(tool_call)
            rprint("news_result", news_result)
            messages_list.append(news_result)
        elif tool_name == "get_stock_price":
            stock_price_result = get_stock_price.invoke(tool_call)
            rprint("stock_price_result", stock_price_result)
            messages_list.append(stock_price_result)
    

final_response = model.invoke(messages_list)
messages_list.append(final_response)
for msg in messages_list:
    msg.pretty_print()

stock_price_result
ToolMessage(
    content='苹果公司 today 价格为 185.2美元',
    name='get_stock_price',
    tool_call_id='call_JyqRzuEpTz6ewVMnJUJlBJdO'
)

news_result
ToolMessage(
    content='苹果发布新款iPhone，股价上涨3%\n苹果与欧盟达成反垄断和解协议\n苹果将在印度扩大生产规模',
    name='search_news',
    tool_call_id='call_1QXmRPAlH5LTXS3ihxmuQ5Re'
)

没有工具调用，直接返回答案
================================ Human Message =================================

苹果公司今天的股价是多少？最近有什么新闻？
================================== Ai Message ==================================

[{'arguments': '{"company":"苹果公司","timeframe":"today"}', 'call_id': 'call_JyqRzuEpTz6ewVMnJUJlBJdO', 'name': 'get_stock_price', 'type': 'function_call', 'id': 'fc_1790415115663_9b0q82mjmoh', 'status': 'completed'}, {'arguments': '{"company":"苹果公司"}', 'call_id': 'call_1QXmRPAlH5LTXS3ihxmuQ5Re', 'name': 'search_news', 'type': 'function_call', 'id': 'fc_1790415115663_nfx7b3vmjk', 'status': 'completed'}]
Tool Calls:
  get_stock_price (call_JyqRzuEpTz6ewVMnJUJlBJdO)
 Call ID: call_JyqRzuEpTz6ewVMnJUJlBJdO
  Args:
    company: 苹果公司
    timeframe: today
  search_news (call_1QXmRPAlH5LTXS3ihxmuQ5Re)
 Call ID: call_1QXmRPAlH5LTXS3ihxmuQ5Re
  Args:
    company: 苹果公司
================================= Tool Message =================================
Name: get_stock_price

苹果公司 today 价格为 185.2美元
========